# Tiff's Experiments — Stain Normalization + Adapter Fine-tuning

**Goal:** Reduce the domain shift between training centers (0, 3, 4) and the unseen val/test centers.

**Best partner result so far:** Virchow2 clsmean + color jitter + outlier filtering → **0.97637**

| Config | Backbone | Stain | Augmentation | Adapter |
|--------|----------|-------|-------------|---------|
| A | Virchow2 clsmean | — | ColorJitter | — (baseline) |
| B | Virchow2 clsmean | Macenko | ColorJitter | — |
| C | Virchow2 clsmean | Macenko | ColorJitter | Bottleneck (class) |
| D | Virchow2 clsmean | — | HED RandAugment | — (ablation) |
| E | Virchow2 clsmean | — | ColorJitter | LoRA (class) |
| F | Virchow2 clsmean | — | ColorJitter | VeRA (class) |

## 0a. Clone repo

Run once per Colab session. Clones from the `tiff` branch so all `utils/` code is available.

In [ ]:
%cd /content
!rm -rf Kaggle-Challenge
!git clone --branch tiff --single-branch https://github.com/lahnabek/Kaggle-Challenge.git
%cd Kaggle-Challenge

## 0b. Download competition data via kagglehub

Add your Kaggle API token to Colab Secrets before running this cell:
1. Click the **key icon** (🔑) in the left sidebar
2. Add a secret named `KAGGLE_TOKEN` with your token as the value
3. Make sure "Notebook access" is toggled on

In [ ]:
import os
import kagglehub

# Load Kaggle token from Colab Secrets (add it in the key icon panel on the left).
# Name the secret KAGGLE_TOKEN and paste your token value there.
try:
    from google.colab import userdata
    os.environ["KAGGLE_TOKEN"] = userdata.get("KAGGLE_TOKEN")
except Exception:
    pass  # running outside Colab — token must already be in env

kagglehub.competition_download('mva-dlmi-2026-histopathology-ood-classification')

## 0c. Mount Google Drive (for saving results)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
GDRIVE_OUT = "/content/drive/MyDrive/dlmi_results"
!mkdir -p "{GDRIVE_OUT}"
print(f"Results will be saved to: {GDRIVE_OUT}")

## 0d. Copy H5 files to repo root

In [ ]:
KAGGLE_DATA = "/root/.cache/kagglehub/competitions/mva-dlmi-2026-histopathology-ood-classification"
!ls -lh "{KAGGLE_DATA}"
!cp "{KAGGLE_DATA}/train.h5" train.h5
!cp "{KAGGLE_DATA}/val.h5"   val.h5
!cp "{KAGGLE_DATA}/test.h5"  test.h5
!ls -lh *.h5

## 0e. Install requirements

In [ ]:
!pip install -r requirements.txt

## 0f. Imports

In [ ]:
import os
import sys
from pathlib import Path

# After %cd Kaggle-Challenge above, cwd is the repo root — just add it to path.
_ROOT = Path.cwd().resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import numpy as np
import torch
import h5py

from utils.config import (
    AugmentationConfig,
    ModelConfig,
    ModuleSpec,
    ProcessingConfig,
    RunConfig,
    TrainConfig,
    EarlyStoppingConfig,
)
from utils.data_augmentation import Compose, make_train_augmentations
from utils.model import DefaultMLPAdapter, LoRAAdapter, VeRAAdapter
from utils.outliers import MethodCOutlierParams
from utils.experiment import run_experiment
from utils.constants import DEVICE, RUNS_DIR
from utils.stain_normalization import MacenkoNormalizer, fit_normalizer_from_h5

print(f"Device  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")
print("Adapters available: DefaultMLPAdapter (Bottleneck), LoRAAdapter, VeRAAdapter")

## 0g. Hugging Face login

Required for Virchow2 (`paige-ai/Virchow2` is a gated model). Set `HF_TOKEN` in Colab Secrets or as an env var.

In [ ]:
import huggingface_hub

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)
    print("HF login: OK")
else:
    print("WARNING: HF_TOKEN not set — Virchow2 will fail.")
    print("Set it via: Colab Secrets panel or os.environ['HF_TOKEN'] = '<your_token>'")

## 1. Fit Macenko stain normalizer

Sample patches from training **center 0** to define the target stain appearance.
All patches (train, val, test) will be transformed to match this reference during preprocessing.

In [ ]:
print("Fitting Macenko normalizer on training center 0 (200 patches)...")
macenko = fit_normalizer_from_h5(
    h5_path="train.h5",
    n_reference_patches=200,
    target_center=0,
    luminosity_threshold=0.8,
    angular_percentile=99.0,
    seed=42,
)
print(f"Stain matrix:\n{macenko.stain_matrix_target_}")
print(f"Max concentrations: {macenko.max_conc_target_}")

In [ ]:
# Sanity check: visualize a raw vs. normalized patch
import matplotlib.pyplot as plt

with h5py.File("train.h5", "r") as f:
    sample_key = list(f.keys())[42]
    raw_img = np.array(f[sample_key]["img"])   # (3, H, W)

norm_img = macenko.transform(raw_img)           # (3, H, W)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(np.moveaxis(raw_img, 0, -1))
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(np.moveaxis(norm_img, 0, -1))
axes[1].set_title("Macenko Normalized")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 2. Define experiment configurations

In [ ]:
# Shared hyperparameters
RESIZE       = (98, 98)   # matches partner's Virchow2 setup in my_pipeline_colab.ipynb
SEEDS        = [0]        # increase to [0, 1, 2] for more stable estimates
N_EPOCHS     = 30
BATCH_SIZE   = 16
OUTLIER_PARAMS = MethodCOutlierParams()

# ---- augmentation factories ----

# Color jitter only — allows embedding precompute (jitter is cheap, stochastic but light)
jitter_only = make_train_augmentations(
    use_color_jitter=True,
    use_he_randaugment=False,
    jitter_brightness=0.2,
    jitter_contrast=0.2,
    jitter_saturation=0.2,
)

# Macenko + color jitter: normalize stain first, then apply stochastic jitter.
# Jitter is stochastic → allows_embedding_precompute=False (online forward pass each epoch).
_jitter = make_train_augmentations(
    use_color_jitter=True,
    use_he_randaugment=False,
    jitter_brightness=0.2,
    jitter_contrast=0.2,
    jitter_saturation=0.2,
)
macenko_and_jitter = Compose(
    (macenko, _jitter),
    allows_embedding_precompute=False,
)

# HED RandAugment (stochastic stain augmentation — no normalization)
hed_augment = make_train_augmentations(
    use_color_jitter=True,
    use_he_randaugment=True,
    rand_n=3,
    rand_m=5.0,
)

print("Augmentation factories ready.")

In [ ]:
# Config A — baseline: Virchow2 + color jitter, no adapter (match partner's submitted config)
config_A = RunConfig(
    run_name="tiff_A_virchow2_jitter_baseline",
    seeds=SEEDS,
    processing=ProcessingConfig(
        resize_hw=RESIZE,
        imagenet_normalize=True,
        extra_transform=jitter_only,
    ),
    model=ModelConfig(
        backbone_name="virchow2_clsmean",
        adapter=ModuleSpec(enabled=False),
    ),
    train=TrainConfig(
        batch_size=BATCH_SIZE,
        lr=1e-3,
        num_epochs=N_EPOCHS,
        early_stopping=EarlyStoppingConfig(monitor="val_loss", patience=7),
        use_center_balanced_batches=True,
        center_sampling="uniform",
    ),
    outlier_params=OUTLIER_PARAMS,
    do_predict_test=True,
)

# Config B — add Macenko stain normalization before the backbone
config_B = RunConfig(
    run_name="tiff_B_virchow2_macenko_jitter",
    seeds=SEEDS,
    processing=ProcessingConfig(
        resize_hw=RESIZE,
        imagenet_normalize=True,
        extra_transform=macenko_and_jitter,
    ),
    model=ModelConfig(
        backbone_name="virchow2_clsmean",
        adapter=ModuleSpec(enabled=False),
    ),
    train=TrainConfig(
        batch_size=BATCH_SIZE,
        lr=1e-3,
        num_epochs=N_EPOCHS,
        early_stopping=EarlyStoppingConfig(monitor="val_loss", patience=7),
        use_center_balanced_batches=True,
        center_sampling="uniform",
    ),
    outlier_params=OUTLIER_PARAMS,
    do_predict_test=True,
)

# Config C — Macenko + Bottleneck adapter (from class); lower LR for adapter fine-tuning
config_C = RunConfig(
    run_name="tiff_C_virchow2_macenko_bottleneck",
    seeds=SEEDS,
    processing=ProcessingConfig(
        resize_hw=RESIZE,
        imagenet_normalize=True,
        extra_transform=macenko_and_jitter,
    ),
    model=ModelConfig(
        backbone_name="virchow2_clsmean",
        adapter=ModuleSpec(
            enabled=True,
            module_cls=DefaultMLPAdapter,
            module_kwargs={"hidden_dim": 256, "dropout": 0.1},
        ),
    ),
    train=TrainConfig(
        batch_size=BATCH_SIZE,
        lr=1e-4,
        num_epochs=N_EPOCHS,
        early_stopping=EarlyStoppingConfig(monitor="val_loss", patience=7),
        use_center_balanced_batches=True,
        center_sampling="uniform",
    ),
    outlier_params=OUTLIER_PARAMS,
    do_predict_test=True,
)

# Config D — HED RandAugment instead of normalization (ablation: augmentation vs. normalization)
config_D = RunConfig(
    run_name="tiff_D_virchow2_hed_augment",
    seeds=SEEDS,
    processing=ProcessingConfig(
        resize_hw=RESIZE,
        imagenet_normalize=True,
        extra_transform=hed_augment,
    ),
    model=ModelConfig(
        backbone_name="virchow2_clsmean",
        adapter=ModuleSpec(enabled=False),
    ),
    train=TrainConfig(
        batch_size=BATCH_SIZE,
        lr=1e-3,
        num_epochs=N_EPOCHS,
        early_stopping=EarlyStoppingConfig(monitor="val_loss", patience=7),
        use_center_balanced_batches=True,
        center_sampling="uniform",
    ),
    outlier_params=OUTLIER_PARAMS,
    do_predict_test=True,
)

# Config E — LoRA adapter (from class): low-rank residual, rank=16
config_E = RunConfig(
    run_name="tiff_E_virchow2_lora",
    seeds=SEEDS,
    processing=ProcessingConfig(
        resize_hw=RESIZE,
        imagenet_normalize=True,
        extra_transform=jitter_only,
    ),
    model=ModelConfig(
        backbone_name="virchow2_clsmean",
        adapter=ModuleSpec(
            enabled=True,
            module_cls=LoRAAdapter,
            module_kwargs={"rank": 16, "lora_alpha": 16.0},
        ),
    ),
    train=TrainConfig(
        batch_size=BATCH_SIZE,
        lr=1e-4,
        num_epochs=N_EPOCHS,
        early_stopping=EarlyStoppingConfig(monitor="val_loss", patience=7),
        use_center_balanced_batches=True,
        center_sampling="uniform",
    ),
    outlier_params=OUTLIER_PARAMS,
    do_predict_test=True,
)

# Config F — VeRA adapter (from class): fixed random matrices + trainable diagonal scaling
# very few trainable params (~2.6K) so we can use a higher LR
config_F = RunConfig(
    run_name="tiff_F_virchow2_vera",
    seeds=SEEDS,
    processing=ProcessingConfig(
        resize_hw=RESIZE,
        imagenet_normalize=True,
        extra_transform=jitter_only,
    ),
    model=ModelConfig(
        backbone_name="virchow2_clsmean",
        adapter=ModuleSpec(
            enabled=True,
            module_cls=VeRAAdapter,
            module_kwargs={"rank": 16, "seed": 42},
        ),
    ),
    train=TrainConfig(
        batch_size=BATCH_SIZE,
        lr=1e-3,
        num_epochs=N_EPOCHS,
        early_stopping=EarlyStoppingConfig(monitor="val_loss", patience=7),
        use_center_balanced_batches=True,
        center_sampling="uniform",
    ),
    outlier_params=OUTLIER_PARAMS,
    do_predict_test=True,
)

ALL_RUNS = [config_A, config_B, config_C, config_D, config_E, config_F]
print(f"Defined {len(ALL_RUNS)} configs")

## 3. Run experiments

Run one config at a time so you can inspect val accuracy before committing to the full sweep.
Results are written to `runs/<run_name>/seed_<k>/metrics.csv`.

In [ ]:
results_A = run_experiment(config_A)

In [ ]:
results_B = run_experiment(config_B)

In [ ]:
results_C = run_experiment(config_C)

In [ ]:
results_D = run_experiment(config_D)

In [ ]:
# Config E — LoRA adapter (low-rank, ~81K params)
results_E = run_experiment(config_E)

In [ ]:
# Config F — VeRA adapter (fixed random matrices + diagonal scaling, ~2.6K params)
results_F = run_experiment(config_F)

## 4. Save results 

In [ ]:
import subprocess

for cfg in ALL_RUNS:
    run_dir = f"runs/{cfg.run_name}"
    import os as _os
    if _os.path.isdir(run_dir):
        zip_name = f"runs_{cfg.run_name}.zip"
        subprocess.run(["zip", "-r", zip_name, run_dir], check=True)
        subprocess.run(["cp", zip_name, GDRIVE_OUT], check=True)
        print(f"Saved {zip_name} to Drive.")
    else:
        print(f"Skipped {cfg.run_name} (not run yet).")

## 5. Compare results

In [ ]:
import pandas as pd

dfs = []
for cfg in ALL_RUNS:
    # metrics.csv may live directly under run_name/ or under run_name/seed_0/
    for csv_path in [
        f"runs/{cfg.run_name}/metrics.csv",
        f"runs/{cfg.run_name}/seed_0/metrics.csv",
    ]:
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            df["run_name"] = cfg.run_name
            dfs.append(df)
            break

if dfs:
    all_metrics = pd.concat(dfs, ignore_index=True)
    summary = (
        all_metrics[all_metrics["split"] == "val"]
        .groupby("run_name")["accuracy"]
        .max()
        .reset_index()
        .rename(columns={"accuracy": "best_val_acc"})
        .sort_values("best_val_acc", ascending=False)
    )
    print(summary.to_string(index=False))
else:
    print("No metrics found yet — run the cells above first.")